In [2]:
"""
# Main Application Pipeline
Purpose: To tie the modules together into a complete end-to-end ATS parsing, embedding, and scoring pipeline.
Architecture: Orchestrates data flow. Reads files -> Parses text -> Preprocesses/Extracts Skills -> Generates Embeddings -> Computes Similarity -> Generates ATS Score -> Provides Recommendations and AI Feedback.
"""

import os
import glob
import importnb
with importnb.Notebook():
    from parser import ResumeParser  # type: ignore
    from preprocessing import NLPPreprocessor  # type: ignore
    from embeddings import EmbeddingEngine  # type: ignore
    from similarity import SimilarityEngine  # type: ignore
    from ats_score import ATSScoreEngine  # type: ignore
    from recommendations import RecommendationEngine  # type: ignore
    from ai_feedback import AIFeedbackLayer  # type: ignore
def main():
    print("==================================================")
    print("        AI Resume ATS Analyzer System             ")
    print("==================================================\n")

    jd_path = "datasets/job_description.txt"
    resume_dir = "resumes/"

    # Step 1: Initialization
    print("[1] Initializing Modules...")
    parser = ResumeParser()
    preprocessor = NLPPreprocessor()
    embedder = EmbeddingEngine()
    similarity_engine = SimilarityEngine()
    score_engine = ATSScoreEngine()
    rec_engine = RecommendationEngine()
    ai_feedback = AIFeedbackLayer()
    print("Modules initialized successfully.\n")

    # Step 2: Parse Job Description (Done once)
    print("[2] Parsing Job Description...")
    jd_text = parser.parse(jd_path)
    if not jd_text:
        print("Error: Could not parse Job Description.")
        return
        
    jd_data = preprocessor.preprocess(jd_text)
    jd_embedding = embedder.generate_embedding(jd_data['lemmatized_text'])
    print(f"Required Skills Found: {len(jd_data['skills'])}\n")

    # Step 3: Batch Process all Resumes
    print("[3] Scanning 'resumes/' directory for candidates...")
    resume_files = [f for f in os.listdir(resume_dir) if os.path.isfile(os.path.join(resume_dir, f))]
    
    if not resume_files:
        print("No resumes found in the 'resumes/' folder. Please add some PDF, DOCX, or TXT files!")
        return
        
    print(f"Found {len(resume_files)} resumes. Processing...\n")
    
    results = []

    for filename in resume_files:
        filepath = os.path.join(resume_dir, filename)
        print(f"--- Analyzing: {filename} ---")
        
        # Extract text
        resume_text = parser.parse(filepath)
        if not resume_text:
            print(f"Skipping {filename} due to parse error.\n")
            continue
            
        # Preprocess and Embed
        resume_data = preprocessor.preprocess(resume_text)
        resume_embedding = embedder.generate_embedding(resume_data['lemmatized_text'])
        
        # Calculate Similarity
        similarity_score = similarity_engine.calculate_similarity(resume_embedding, jd_embedding)
        
        # Generate ATS Score
        ats_result = score_engine.generate_score(
            similarity_score=similarity_score,
            resume_skills=resume_data['skills'],
            jd_skills=jd_data['skills'],
            resume_text=resume_text
        )
        
        score = ats_result['final_score']
        print(f"Candidate ATS Score: {score} / 100")
        print(f"Skills Found: {len(resume_data['skills'])}")
        print("-" * 50)
        
        # Generate Recommendations
        recs = rec_engine.generate_recommendations(
            resume_skills=resume_data['skills'],
            jd_skills=jd_data['skills']
        )

        # Store for ranking later
        results.append({
            "filename": filename,
            "score": score,
            "skills": resume_data['skills'],
            "ats_result": ats_result,
            "recommendations": recs,
            "text": resume_text
        })
        
    # Final Ranking
    print("\n==================================================")
    print("                 FINAL ATS RANKING                ")
    print("==================================================")

    # Sort results highest score first
    results.sort(key=lambda x: x['score'], reverse=True)

    for rank, res in enumerate(results, 1):
        print(f"#{rank} | {res['filename']} | Score: {res['score']} / 100")

    # â”€â”€ Detailed Per-Candidate Report â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print("\n")
    print("==================================================")
    print("          DETAILED CANDIDATE REPORTS              ")
    print("==================================================")

    for rank, res in enumerate(results, 1):
        recs  = res['recommendations']
        breakdown = res['ats_result']['breakdown']

        print(f"\n{'='*52}")
        print(f"  RANK #{rank}  |  {res['filename']}")
        print(f"{'='*52}")

        # â”€â”€ Score Breakdown â”€â”€
        print(f"\n  FINAL ATS SCORE : {res['score']} / 100")
        print(f"  {'-'*40}")
        print(f"  Semantic Similarity Score : {breakdown['semantic_score']:>6.2f} / 100  (weight 50%)")
        print(f"  Keyword Match Score       : {breakdown['keyword_score']:>6.2f} / 100  (weight 40%)")
        print(f"  Structure / Length Score  : {breakdown['structure_score']:>6.2f} / 100  (weight 10%)")

        # â”€â”€ Skills Found â”€â”€
        print(f"\n  SKILLS DETECTED ({len(res['skills'])}) :")
        if res['skills']:
            skill_line = ", ".join(sorted(res['skills']))
            print(f"  {skill_line}")
        else:
            print("  None detected.")

        # â”€â”€ Missing Keywords â”€â”€
        print(f"\n  MISSING KEYWORDS ({len(recs['missing_keywords'])}) :")
        if recs['missing_keywords']:
            for kw in sorted(recs['missing_keywords']):
                print(f"    [x]  {kw}")
        else:
            print("    None - full keyword coverage!")

        # â”€â”€ Bonus Keywords â”€â”€
        print(f"\n  BONUS KEYWORDS ({len(recs['bonus_keywords'])}) :")
        if recs['bonus_keywords']:
            for kw in sorted(recs['bonus_keywords']):
                print(f"    [+]  {kw}")
        else:
            print("    None.")

        # â”€â”€ Actionable Recommendations â”€â”€
        print(f"\n  RECOMMENDATIONS :")
        for i, tip in enumerate(recs['actionable_feedback'], 1):
            print(f"    {i}. {tip}")

        print()

    print("==================================================")
    print("  Run complete.")
    print("==================================================")

if __name__ == "__main__":
    os.makedirs("resumes", exist_ok=True)
    os.makedirs("datasets", exist_ok=True)
    main()


        AI Resume ATS Analyzer System             

[1] Initializing Modules...
Loading embedding model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modules initialized successfully.

[2] Parsing Job Description...
Required Skills Found: 12

[3] Scanning 'resumes/' directory for candidates...
Found 1 resumes. Processing...

--- Analyzing: Sayan Resume 1..2 .pdf ---
Candidate ATS Score: 62.22 / 100
Skills Found: 8
--------------------------------------------------

                 FINAL ATS RANKING                
#1 | Sayan Resume 1..2 .pdf | Score: 62.22 / 100


          DETAILED CANDIDATE REPORTS              

  RANK #1  |  Sayan Resume 1..2 .pdf

  FINAL ATS SCORE : 62.22 / 100
  ----------------------------------------
  Semantic Similarity Score :  64.44 / 100  (weight 50%)
  Keyword Match Score       :  50.00 / 100  (weight 40%)
  Structure / Length Score  : 100.00 / 100  (weight 10%)

  SKILLS DETECTED (8) :
  git, machine learning, nlp, numpy, pandas, python, spacy, sql

  MISSING KEYWORDS (6) :
    [x]  aws
    [x]  docker
    [x]  kubernetes
    [x]  pytorch
    [x]  rest
    [x]  tensorflow

  BONUS KEYWORDS (2) :
   